In [ ]:

import numpy as np
import pandas as pd



import sys
from pathlib import Path

PROJECT_ROOT = next(d for d in (Path.cwd(), *Path.cwd().parents)
                    if (d / "config.yaml").exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import embedding_name, load_config
from src.evaluation import standard_evaluations, write_evaluation

CFG = load_config(PROJECT_ROOT)

METHOD = CFG["vpr"]["method"]
RETRIEVAL_METHOD = CFG["retrieval"]["method"]
ADAPTER = CFG["vpr"].get("adapter", "none")
RESULT_DIR = PROJECT_ROOT / "results" 
EMBEDDING_DIR = PROJECT_ROOT / "data" / "embeddings" / METHOD
RETRIEVAL_DIR = RESULT_DIR / "retrieval"


EMBEDDING_NAME = embedding_name(CFG)


embedding_path = EMBEDDING_DIR / f"{EMBEDDING_NAME}_embeddings.npy"
metadata_path = EMBEDDING_DIR / f"{EMBEDDING_NAME}_metadata.parquet"

RETRIEVAL_DIR.mkdir(parents = True, exist_ok = True)

embedding_metadata = pd.read_parquet(metadata_path)
database_mask = (embedding_metadata["split"] == "database").to_numpy()
query_mask = (embedding_metadata["split"] == "query").to_numpy()
database_metadata = embedding_metadata[database_mask].reset_index(drop=True)
query_metadata = embedding_metadata[query_mask].reset_index(drop=True)

# Nur der Reihe nach gelesen -- fuer die Probe unten und die Dimension.
# Vollstaendig laden waere bei MegaLoc 11 GB fuer 256 Zeilen.
embeddings = np.load(embedding_path, mmap_mode="r")
DIM = int(embeddings.shape[1])
database_rows = np.flatnonzero(database_mask)
query_rows = np.flatnonzero(query_mask)

from src.run_guard import embedding_fingerprint, require_fingerprint, print_run_header, validate_config

validate_config(CFG)
print_run_header(CFG, "07_evaluation")

FINGERPRINT = embedding_fingerprint(CFG, METHOD, ADAPTER, embedding_metadata)

retrieval_path = RETRIEVAL_DIR / METHOD / f"{EMBEDDING_NAME}_retrieval.npz"

require_fingerprint(embedding_path, FINGERPRINT, what="Embeddings")
require_fingerprint(retrieval_path, FINGERPRINT, what="Retrieval-Ergebnis")

retrieval = np.load(retrieval_path)
retrieved_indices = retrieval["indices"]
similarities = retrieval["similarities"]


# Der Fingerabdruck oben deckt die config ab, nicht den Dateiinhalt. Diese
# Probe rechnet gespeicherte Similarities nach und schlaegt an, wenn 06
# nach einer Aenderung an den Embeddings nicht neu gelaufen ist.

_probe = np.sort(np.random.default_rng(0).choice(
    len(query_rows), min(256, len(query_rows)), replace=False
))
_expected = (
    np.asarray(embeddings[query_rows[_probe]], dtype=np.float32)
    * np.asarray(embeddings[database_rows[retrieved_indices[_probe, 0]]], dtype=np.float32)
).sum(axis=1)

if not np.allclose(_expected, similarities[_probe, 0], atol=1e-4):
    raise RuntimeError(
        f"{retrieval_path.name} passt nicht zu {embedding_path.name}.\n"
        f"  gespeicherte Similarity (Mittel): {similarities[_probe, 0].mean():.4f}\n"
        f"  aus den Embeddings gerechnet:     {_expected.mean():.4f}\n"
        f"  -> 06_retrieval mit frischem Kernel neu ausfuehren."
    )

print(f"Retrieval geprueft: {retrieval_path.name} passt zu {embedding_path.name}")


In [ ]:
# Die Rechnung steht in src/evaluation.py -- dieselbe fuer 06 und fuer jedes
# Experiment, das Trefferlisten umsortiert. Vier Ground-Truth-Varianten,
# Zufallsbasis inklusive.
befunde = standard_evaluations(retrieved_indices, query_metadata, database_metadata, CFG)

EVAL_PATH = write_evaluation(
    RESULT_DIR / "evaluation" / f"{EMBEDDING_NAME}.json",
    CFG, EMBEDDING_NAME, DIM, len(database_metadata), befunde,
    fingerprint=FINGERPRINT, root=PROJECT_ROOT,
)
print(f"\nAuswertung gespeichert: {EVAL_PATH}")